# Federalist Papers: Hauptkomponentenanalyse

Ergänzendes historisches Beispiel auf Grundlage der Exploration von Kai S. Kurono.

Vorbereitung: `python prepare_data.py --federalist`. Beide Notebooks verwenden denselben Parser in `stylometry/federalist.py`. Er entfernt Titel, Publikationsdaten, Autorenangaben, Anrede und PUBLIUS-Signatur samt nachfolgenden Endnoten. Von Aufsatz 70 bleibt die erste Fassung erhalten. Die Nummern stammen aus den Überschriften.

Die zwölf Aufsätze 49–58, 62 und 63 erhalten die Kategorie `Disputed`, die Aufsätze 18–20 die Kategorie `Joint`. Diese Kategorien dienen der Darstellung. Es handelt sich um eine explorative Visualisierung ohne Testklassifikation.

Die bereinigten Abbildungen ersetzen die früheren Darstellungen. Die PCA nutzt TF-IDF. Die HCA verwendet weiterhin Worthäufigkeiten mit deaktivierter IDF.

## 1. Bereinigte Texte und getrennte Metadaten

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from stylometry.federalist import load_federalist, COLORS

papers = pd.DataFrame(load_federalist())
display(papers.groupby('author').size().rename('Essays').to_frame())
assert papers.number.tolist() == list(range(1, 86))

## 2. Repräsentation und Berechnung

In [ ]:
from sklearn.decomposition import PCA

vectorizer = TfidfVectorizer(max_features=1000, use_idf=True)
vectors = vectorizer.fit_transform(papers.body).toarray()
pca = PCA(n_components=2, svd_solver='full')
coordinates = pca.fit_transform(vectors)
print('Explained variance:', pca.explained_variance_ratio_)

## 3. Darstellung und Interpretation

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
for author, color in COLORS.items():
    mask = papers.author.eq(author).to_numpy()
    marker = '^' if author == 'Disputed' else 's' if author == 'Joint' else 'o'
    ax.scatter(coordinates[mask, 0], coordinates[mask, 1], c=color, marker=marker, label=author, alpha=.8)
for number, (x, y) in zip(papers.number, coordinates):
    ax.annotate(str(number), (x, y), xytext=(4, 4), textcoords='offset points', fontsize=7)
ax.set(xlabel=f'PC1 ({pca.explained_variance_ratio_[0]:.1%})', ylabel=f'PC2 ({pca.explained_variance_ratio_[1]:.1%})', title='Federalist Papers: text bodies, TF-IDF and PCA')
ax.legend()
ax.grid(alpha=.15)
fig.tight_layout()
fig.savefig('federalist_pca.png', dpi=160)
plt.show()

# Largest absolute loadings are directions in this projection, not classifier importance.
features = vectorizer.get_feature_names_out()
for component in range(2):
    order = np.argsort(np.abs(pca.components_[component]))[-10:][::-1]
    display(pd.DataFrame({'feature': features[order], 'loading': pca.components_[component, order]}))

Die Autorenfarben sind bekannte Metadaten. Sichtbare Gruppen beweisen keine zuverlässige Autorschaftserkennung. Die historischen Texte unterscheiden sich in Epoche und Genre von Reuters. Quellen: [Project Gutenberg](https://www.gutenberg.org/ebooks/18), [SciPy linkage](https://docs.scipy.org/doc/scipy/reference/generated/scipy.cluster.hierarchy.linkage.html).